In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🔧 Libraries imported successfully!")


In [ ]:
# Load the scraped properties data
try:
    df = pd.read_csv('outputs/scraped_properties.csv')
    print(f"✅ Loaded {len(df)} properties from CSV")
    print(f"📅 Data shape: {df.shape}")
except FileNotFoundError:
    print("❌ No scraped_properties.csv found. Please run the scraper first.")
    print("💡 You can run: python src/scraprop.py")
    df = pd.DataFrame()  # Empty dataframe to avoid errors
except Exception as e:
    print(f"❌ Error loading data: {e}")
    df = pd.DataFrame()


In [ ]:
# Display basic information about the dataset
if not df.empty:
    print("📋 Dataset Overview:")
    print(f"Total properties: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print("\n📊 Data types:")
    print(df.dtypes)
    
    # Show first few rows
    print("\n🔍 First 3 rows:")
    display(df.head(3))


In [ ]:
if not df.empty:
    # Parse LLM analysis from JSON strings if stored as strings
    if 'llm_analysis' in df.columns and df['llm_analysis'].dtype == 'object':
        def safe_parse_json(x):
            if pd.isna(x) or x == '':
                return {}
            try:
                if isinstance(x, str):
                    return json.loads(x)
                return x
            except:
                return {}
        
        df['llm_analysis_parsed'] = df['llm_analysis'].apply(safe_parse_json)
    
    # Extract key features from LLM analysis
    if 'llm_analysis_parsed' in df.columns:
        df['ai_neighbourhood'] = df['llm_analysis_parsed'].apply(lambda x: x.get('neighbourhood', 'N/A') if isinstance(x, dict) else 'N/A')
        df['ai_ground_floor'] = df['llm_analysis_parsed'].apply(lambda x: x.get('is_ground_floor', False) if isinstance(x, dict) else False)
        df['ai_outdoor_space'] = df['llm_analysis_parsed'].apply(lambda x: x.get('has_outdoor_space', False) if isinstance(x, dict) else False)
        df['ai_outdoor_type'] = df['llm_analysis_parsed'].apply(lambda x: x.get('outdoor_space_type', 'none') if isinstance(x, dict) else 'none')
        df['ai_near_avenue'] = df['llm_analysis_parsed'].apply(lambda x: x.get('near_important_avenue', False) if isinstance(x, dict) else False)
        df['ai_near_transport'] = df['llm_analysis_parsed'].apply(lambda x: x.get('near_subway_train', False) if isinstance(x, dict) else False)
        df['ai_price_numeric'] = df['llm_analysis_parsed'].apply(lambda x: x.get('price_numeric', 0) if isinstance(x, dict) else 0)
        df['ai_surface_m2'] = df['llm_analysis_parsed'].apply(lambda x: x.get('surface_m2', 0) if isinstance(x, dict) else 0)
    
    # Clean and convert price/surface columns
    if 'ai_price_numeric' in df.columns:
        df['price_cleaned'] = pd.to_numeric(df['ai_price_numeric'], errors='coerce').fillna(0)
    
    if 'ai_surface_m2' in df.columns:
        df['surface_cleaned'] = pd.to_numeric(df['ai_surface_m2'], errors='coerce').fillna(0)
    
    # Calculate price per m²
    if 'price_cleaned' in df.columns and 'surface_cleaned' in df.columns:
        df['price_per_m2'] = np.where(df['surface_cleaned'] > 0, 
                                     df['price_cleaned'] / df['surface_cleaned'], 
                                     np.nan)
    
    print("✅ Data preprocessing completed!")
    print(f"New columns added: {[col for col in df.columns if col.startswith('ai_') or col.endswith('_cleaned') or col == 'price_per_m2']}")


In [ ]:
if not df.empty:
    print("📊 KEY STATISTICS")
    print("=" * 50)
    
    # Score distribution
    if 'score' in df.columns:
        print(f"\n⭐ SCORES:")
        print(f"   Average score: {df['score'].mean():.1f}")
        print(f"   Median score: {df['score'].median():.1f}")
        print(f"   Max score: {df['score'].max():.1f}")
        print(f"   Min score: {df['score'].min():.1f}")
        print(f"   Properties with score > 20: {len(df[df['score'] > 20])}")
    
    # Display top 5 properties by score
    print(f"\n🏆 TOP 5 PROPERTIES BY SCORE:")
    top_5 = df.nlargest(5, 'score') if 'score' in df.columns else df.head(5)
    for i, (idx, row) in enumerate(top_5.iterrows()):
        print(f"{i+1}. Score: {row.get('score', 'N/A'):.0f} - {row.get('llm_neighbourhood', 'N/A')} - {row.get('url', 'N/A')}")


In [ ]:
def filter_properties(df, min_score=0, max_price=None, min_surface=0, neighborhoods=None, 
                     ground_floor=None, outdoor_space=None, near_transport=None, near_avenue=None):
    """Filter properties based on various criteria."""
    filtered = df.copy()
    
    # Score filter
    if 'score' in filtered.columns:
        filtered = filtered[filtered['score'] >= min_score]
    
    # Price filter (using llm_price_numeric column)
    if max_price and 'llm_price_numeric' in filtered.columns:
        filtered = filtered[filtered['llm_price_numeric'] <= max_price]
    
    # Surface filter
    if min_surface and 'llm_surface_m2' in filtered.columns:
        filtered = filtered[filtered['llm_surface_m2'] >= min_surface]
    
    # Neighborhood filter
    if neighborhoods and 'llm_neighbourhood' in filtered.columns:
        if isinstance(neighborhoods, str):
            neighborhoods = [neighborhoods]
        filtered = filtered[filtered['llm_neighbourhood'].isin(neighborhoods)]
    
    # Boolean feature filters
    if ground_floor is not None and 'llm_is_ground_floor' in filtered.columns:
        filtered = filtered[filtered['llm_is_ground_floor'] == ground_floor]
    
    if outdoor_space is not None and 'llm_has_outdoor_space' in filtered.columns:
        filtered = filtered[filtered['llm_has_outdoor_space'] == outdoor_space]
    
    if near_transport is not None and 'llm_near_subway_train' in filtered.columns:
        filtered = filtered[filtered['llm_near_subway_train'] == near_transport]
    
    if near_avenue is not None and 'llm_near_important_avenue' in filtered.columns:
        filtered = filtered[filtered['llm_near_important_avenue'] == near_avenue]
    
    return filtered

def display_properties(filtered_df, top_n=10, sort_by='score'):
    """Display properties in a formatted way."""
    if filtered_df.empty:
        print("No properties match your criteria.")
        return
    
    # Sort by specified column
    if sort_by in filtered_df.columns:
        df_sorted = filtered_df.sort_values(sort_by, ascending=False)
    else:
        df_sorted = filtered_df
    
    # Display top properties
    for i, (idx, row) in enumerate(df_sorted.head(top_n).iterrows()):
        print(f"\n🏠 PROPERTY #{i+1}")
        print("=" * 60)
        
        # Basic info
        print(f"⭐ Score: {row.get('score', 'N/A')}")
        print(f"🔗 URL: {row.get('url', 'N/A')}")
        print(f"📍 Neighborhood: {row.get('llm_neighbourhood', 'N/A')}")
        print(f"💰 Price: ${row.get('llm_price_numeric', 0):,.0f}")
        print(f"📏 Surface: {row.get('llm_surface_m2', 0):.0f}m²")
        
        # Features
        features = []
        if row.get('llm_is_ground_floor', False):
            features.append('🌳 Ground Floor')
        if row.get('llm_has_outdoor_space', False):
            features.append(f'🌿 Outdoor Space ({row.get("llm_outdoor_space_type", "unknown")})')
        if row.get('llm_near_important_avenue', False):
            features.append('🛣️ Near Avenue')
        if row.get('llm_near_subway_train', False):
            features.append('🚇 Near Transport')
        
        if features:
            print(f"✨ Features: {', '.join(features)}")

print("🎯 Property filtering functions are ready!")


In [ ]:
# Example: Find high-scoring properties
if not df.empty:
    print("🔍 HIGH-SCORING PROPERTIES (Score >= 15):")
    print("-" * 50)
    
    high_score_props = filter_properties(df, min_score=15)
    display_properties(high_score_props, top_n=5)
    
    print(f"\n📊 Found {len(high_score_props)} high-scoring properties total")


In [ ]:
# CUSTOM FILTER - Modify these parameters to find your ideal properties!
if not df.empty:
    print("🛠️ CUSTOM PROPERTY SEARCH")
    print("=" * 50)
    print("Modify the parameters below to find your ideal properties:")
    
    # YOUR CUSTOM CRITERIA HERE - MODIFY AS NEEDED!
    custom_results = filter_properties(df, 
        min_score=10,              # Minimum score
        max_price=800000,          # Maximum price in dollars
        min_surface=50,            # Minimum surface in m²
        neighborhoods=None,        # List of specific neighborhoods or None for all
        ground_floor=None,         # True, False, or None
        outdoor_space=None,        # True, False, or None
        near_transport=None,       # True, False, or None
        near_avenue=None           # True, False, or None
    )
    
    print(f"\n🎯 YOUR CUSTOM SEARCH RESULTS:")
    display_properties(custom_results, top_n=10)
    
    print(f"\n📊 Found {len(custom_results)} properties matching your criteria")
